# Stage 1 Fine-tuning: NLLB-200 for Literal Translation (AI/ML Domain)

This notebook contains the complete code to fine-tune the `facebook/nllb-200-distilled-600M` model on English-Vietnamese machine translation datasets, focusing on the AI/ML textbook domain. Run this notebook on **Kaggle GPU (T4 or P100)**.

In [ ]:
# 1. Install required libraries
!pip install -q transformers datasets accelerate evaluate sacrebleu sentencepiece huggingface_hub

In [ ]:
# 2. Login to Hugging Face Hub
# Note: Alternatively, you can add your HF token as a Kaggle Secret with the key 'HF_TOKEN' and retrieve it via os.getenv('HF_TOKEN')
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Load pre-trained model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

In [ ]:
# 4. Prepare and tokenize translation dataset
# Note: In a real training session, you can load datasets from Hugging Face Hub (e.g. PhoMT, OPUS) 
# combined with your custom collected AI sentences.
from datasets import load_dataset

# For demonstration, we load a small subset of parallel English-Vietnamese text
# Replace this with your actual target dataset path or repository name
try:
    raw_datasets = load_dataset("wmt15", "en-fr", split="train[:1000]") # Placeholder
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Dataset loading failed: {e}. You should upload your own translation dataset.")

In [ ]:
# 5. Data preprocessing function
max_input_length = 128
max_target_length = 128
src_lang = "eng_Latn"
tgt_lang = "vie_Latn"

def preprocess_function(examples):
    # Configure tokenizer language codes
    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang
    
    # Assuming columns 'english' and 'vietnamese'
    inputs = [ex for ex in examples["english"]]
    targets = [ex for ex in examples["vietnamese"]]
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
# tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

In [ ]:
# 6. Configure Seq2Seq training arguments
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="nllb-envi-ai-textbook-stage1",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  # Enable mixed-precision acceleration on GPU
    push_to_hub=True,  # Automatically upload model checkpoint to HF Hub
    hub_model_id="your_hf_username/nllb-envi-ai-textbook-stage1"  # Update with your HF username
)

In [ ]:
# 7. Setup Trainer and start training
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     tokenizer=tokenizer,
#     data_collator=data_collator,
# )

# print("Starting training...")
# trainer.train()

In [ ]:
# 8. Push final model and tokenizer to Hugging Face Hub
# trainer.push_to_hub()